# Silver: parse the PDFs with `ai_parse_document`

Task `silver_parse` of the job `ingest_financebench`. Runs the AI parser on every part in `bronze_pdf_parts` and
saves the result in `silver_parsed`, **one row per part**: `path`, `part_no`, `page_offset`, `parsed` and `parsed_at`.

`parsed` is the parser's output (a VARIANT): the `elements` of the part in reading order, each with a `type`
(title, section_header, text, table as HTML, page_header, ...), its `content` and its page in `bbox[0].page_id`. The
real page of an element in the original PDF is `page_offset + page_id + 1`.

**A stream.** The parts are read as a stream (`readStream`) with `trigger(availableNow=True)`: the run processes every
part it has not seen, `pdfs_per_batch` PDFs at a time (bronze writes one file per PDF), and stops. The checkpoint
remembers what was read, so a rerun only parses new parts, and a saved part is never paid for twice. A few PDFs at a time
also keeps the parser from rate-limiting, which is why bronze cuts long PDFs into parts of 50 pages.

**Only complete parses are saved.** The parser sometimes answers pages with an error (`Request rate limit exceeded`,
`Internal error`), and a part with missing pages would silently lose text. So a part is saved only if it has elements and
no `error_status`.

**A part that was not saved is not retried by a normal run**, because the stream has read it once. To parse those parts
again, run with `retry_failed` = `true`: it deletes the checkpoint, so the stream reads every part again, and the parts
already in `silver_parsed` are skipped, so nothing is paid for twice. The two Intel 8-Ks fail every time
(`Document conversion failed`). The last cell lists what is missing.

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("pdfs_per_batch", "2")  # PDFs parsed at once, small so the parser's rate limit is not hit
dbutils.widgets.dropdown("retry_failed", "false", ["false", "true"])  # true: read every part again (saved ones are skipped)

catalog = dbutils.widgets.get("catalog")
parts_table = f"{catalog}.{dbutils.widgets.get('bronze_schema')}.bronze_pdf_parts"
parsed_table = f"{catalog}.{dbutils.widgets.get('silver_schema')}.silver_parsed"
pdfs_per_batch = int(dbutils.widgets.get("pdfs_per_batch"))

silver_schema = dbutils.widgets.get("silver_schema")
checkpoint = f"/Volumes/{catalog}/{silver_schema}/checkpoints/parse"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{silver_schema}.checkpoints")  # where the stream remembers what it read

In [ ]:
if dbutils.widgets.get("retry_failed") == "true":
    dbutils.fs.rm(checkpoint, True)  # the stream will read every part again

parts = spark.readStream.option("maxFilesPerTrigger", pdfs_per_batch).table(parts_table)
if spark.catalog.tableExists(parsed_table):  # skip the parts that are saved (the first run creates the table below)
    parts = parts.join(spark.table(parsed_table).select("path", "part_no"), ["path", "part_no"], "left_anti")

# The parse is billed and runs once: the filter and the saved column both read the same parser call.
(
    parts.select("path", "part_no", "page_offset", F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"))
    .filter(
        "coalesce(size(cast(parsed:document:elements AS array<variant>)), 0) > 0"  # it has elements
        " AND coalesce(size(cast(parsed:error_status AS array<variant>)), 0) = 0"  # and no page failed
    )
    .withColumn("parsed_at", F.current_timestamp())
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(parsed_table)
    .awaitTermination()
)

In [ ]:
# What is saved, and what is missing (not saved: run with retry_failed = true to parse it again).
display(spark.sql(f"""
SELECT (SELECT count(*) FROM {parts_table}) AS parts, (SELECT count(*) FROM {parsed_table}) AS parts_parsed
"""))
display(spark.sql(f"""
SELECT p.path, p.part_no, p.pages
FROM {parts_table} p LEFT ANTI JOIN {parsed_table} s ON s.path = p.path AND s.part_no = p.part_no
ORDER BY p.path, p.part_no
"""))